# 01 · Specs v3 e descritores multialtitude

Gera as specs v3 do local (grade de 25 m, K = 4 variantes) com `dal.v3.sensor.specs` e **confere os hashes contra `colab/specs/v3_manifest.json`** (se não baterem, o job para). Depois extrai DINOv2 (GeM e CLS) e a qualidade de cada recorte (fração válida e de nuvem) em `MyDrive/drone_v3/features/v3/<modelo>/<spec>/`.

Antes: rode `00_setup.ipynb`. O job é **retomável**: se a sessão cair, rode o setup e este notebook de novo com o mesmo `JOB_ID`.

In [ ]:
JOB_ID = 'v3_extract_taquari_100m'
import pandas as pd
jobs = pd.read_csv('/content/repo/colab/jobs_v3.csv')
job = jobs.set_index('job_id').loc[JOB_ID]
JOB_CONFIG = job.config_path
print(job)

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'dal.v3.descriptors.extract_site', '--job-config', JOB_CONFIG, '--job-id', JOB_ID,
       '--out-root', '/content/drive/MyDrive/drone_v3', '--colab-out', 'colab_out']
print(subprocess.run(cmd, cwd='/content/repo', check=True).returncode)

In [ ]:
import json
done = json.load(open(f'/content/repo/colab_out/{JOB_ID}/DONE.json'))
print(json.dumps(done, indent=1))
main_number = str(done['metrics'].get('crops_per_second', ''))[:12]
subprocess.run([sys.executable, 'scripts/colab_publish_v3.py', '--job-id', JOB_ID,
                '--main-number', f'crops/s {main_number}'], cwd='/content/repo', check=True)